# AdamW — Decoupled Weight Decay Regularization (2017)

_Papers · Foundations & Optimization_

**Apply weight decay straight to the weights — not folded into the gradient — so Adam regularizes correctly and generalizes better.**

---

This notebook is a practice scaffold for the **AdamW — Decoupled Weight Decay Regularization (2017)** lesson. We build AdamW from scratch one piece at a time: the decoupled update, a check against PyTorch, a hand-traced step, and a direct contrast with the Adam+L2 it replaces. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch

### Step 1 — Write the AdamW update from scratch

AdamW runs the usual Adam machinery (smoothed first and second moments, bias correction) on the **loss gradient only**. The key difference from Adam+L2 is the final line: the weight-decay term `wd*theta` is added to the update *directly* and is **not** routed through `m` and `v`. We also set up a fixed positive-definite quadratic loss so the gradient `A·p - b` is deterministic and reproducible.

In [ ]:
import torch

torch.manual_seed(0)

# ---- AdamW update FROM SCRATCH (Algorithm 2, Loshchilov & Hutter 2017) ----
def adamw_step(theta, m, v, g, t, lr=1e-2, beta1=0.9, beta2=0.999, eps=1e-8, wd=0.05):
    """One AdamW step. g is the LOSS gradient only (no lambda*theta folded in)."""
    m = beta1 * m + (1 - beta1) * g           # 1st moment  (momentum)
    v = beta2 * v + (1 - beta2) * g * g       # 2nd moment  (RMSprop term)
    mhat = m / (1 - beta1 ** t)               # bias correction
    vhat = v / (1 - beta2 ** t)
    adaptive = mhat / (torch.sqrt(vhat) + eps)            # the Adam adaptive step
    # decoupled decay: wd*theta is added to the update directly, NOT through m,v
    theta = theta - lr * (adaptive + wd * theta)
    return theta, m, v

# A fixed quadratic loss so the gradient is deterministic and reproducible.
A = torch.randn(4, 4)
A = A @ A.t() + torch.eye(4)                  # symmetric positive-definite
b = torch.randn(4)

def loss_fn(p):
    return 0.5 * (p @ A @ p) - b @ p          # grad = A p - b

theta0 = torch.randn(4)
LR, WD, STEPS = 1e-2, 0.05, 7

### Step 2 — Check it against PyTorch's own AdamW

We race our from-scratch update against `torch.optim.AdamW` from the same start, same hyperparameters, and the same seven steps. Each iteration we recompute the loss gradient with autograd and feed it to `adamw_step`. If the decoupled decay is implemented correctly, the two weight vectors should agree to within `1e-6`.

In [ ]:
# ---- THE ORACLE: my AdamW must equal torch.optim.AdamW over several steps ----
p = theta0.clone().requires_grad_(True)
opt = torch.optim.AdamW([p], lr=LR, betas=(0.9, 0.999), eps=1e-8, weight_decay=WD)
for _ in range(STEPS):
    opt.zero_grad()
    loss_fn(p).backward()
    opt.step()

theta = theta0.clone()
m = torch.zeros(4)
v = torch.zeros(4)
for t in range(1, STEPS + 1):
    th = theta.clone().requires_grad_(True)
    loss_fn(th).backward()                    # autograd gives the loss gradient
    g = th.grad
    theta, m, v = adamw_step(theta, m, v, g, t, lr=LR, wd=WD)

print("torch.optim.AdamW :", p.detach())
print("my AdamW          :", theta)
print("allclose (atol=1e-6):", torch.allclose(theta, p.detach(), atol=1e-6))  # expect True

### Step 3 — Recompute the one-step worked example by hand

A single step makes the decoupling visible. Starting from `theta=0.5` with gradient `g=0.2`, `lr=0.1`, and `wd=0.01`, the adaptive part contributes one piece and the decay term `wd*theta` another, giving `theta_new ≈ 0.3995`. We confirm the same number against `torch.optim.AdamW` on one step.

In [ ]:
# ---- recompute the worked example: one step, theta=0.5, g=0.2 ----
th = torch.tensor([0.5])
m0 = torch.zeros(1)
v0 = torch.zeros(1)
th1, _, _ = adamw_step(th, m0, v0, torch.tensor([0.2]), t=1, lr=0.1, wd=0.01)
print("worked example theta_new:", round(th1.item(), 4))  # 0.3995

pp = torch.tensor([0.5], requires_grad=True)
o2 = torch.optim.AdamW([pp], lr=0.1, weight_decay=0.01)
o2.zero_grad()
pp.grad = torch.tensor([0.2])
o2.step()
print("torch one-step          :", round(pp.detach().item(), 4))  # 0.3995

### Step 4 — Contrast with Adam+L2 (the coupled version)

Finally we build the optimizer AdamW was designed to replace. Adam+L2 folds the decay **into the gradient** (`g = g + wd*theta`) *before* computing the moments — so the decay gets divided by `sqrt(vhat)` like everything else. Running it on the same problem produces different weights, confirming the two are genuinely distinct algorithms.

In [ ]:
# ---- contrast: Adam+L2 folds decay into the gradient (a DIFFERENT optimizer) ----
def adam_l2_step(theta, m, v, g, t, lr=1e-2, beta1=0.9, beta2=0.999, eps=1e-8, wd=0.05):
    g = g + wd * theta                         # decay folded into the gradient (the bug)
    m = beta1 * m + (1 - beta1) * g
    v = beta2 * v + (1 - beta2) * g * g
    mhat = m / (1 - beta1 ** t)
    vhat = v / (1 - beta2 ** t)
    theta = theta - lr * (mhat / (torch.sqrt(vhat) + eps))
    return theta, m, v

theta = theta0.clone()
m = torch.zeros(4)
v = torch.zeros(4)
for t in range(1, STEPS + 1):
    th = theta.clone().requires_grad_(True)
    loss_fn(th).backward()
    theta, m, v = adam_l2_step(theta, m, v, th.grad, t, lr=LR, wd=WD)

print("Adam+L2 differs from AdamW:", not torch.allclose(theta, p.detach(), atol=1e-6))  # expect True

## Visualize the data & results

_Two weights both start at 3.0; we decay them toward 0 with the same lambda. One weight constantly receives LARGE loss-gradients, the other TINY ones. Does Adam+L2 shrink them equally — or does it regularize the small-gradient weight differently than AdamW does?_

### Step 1 — Set up the two-weight decay experiment

The `run` function decays two weights that both start at 3.0 with the same `lambda`. The twist: one weight constantly sees a **large** loss-gradient and the other a **tiny** one. The `decoupled` flag switches between AdamW (decay applied straight to the weight) and Adam+L2 (decay folded into the gradient), so we can watch how each treats the small-gradient weight.

In [ ]:
import torch
torch.manual_seed(1)

def run(decoupled, steps=40):
    th = torch.tensor([3.0, 3.0])             # two weights to decay toward 0
    m = torch.zeros(2)
    v = torch.zeros(2)
    lr, b1, b2, eps, wd = 0.1, 0.9, 0.999, 1e-8, 0.1
    coord1 = []                               # track the SMALL-gradient weight
    for t in range(1, steps + 1):
        g = torch.tensor([1.0, 0.02])         # coord0: big grad, coord1: tiny grad
        if not decoupled:
            g = g + wd * th                   # Adam+L2: fold decay into the gradient
        m = b1 * m + (1 - b1) * g
        v = b2 * v + (1 - b2) * g * g
        mhat = m / (1 - b1 ** t)
        vhat = v / (1 - b2 ** t)
        adaptive = mhat / (torch.sqrt(vhat) + eps)
        if decoupled:
            th = th - lr * (adaptive + wd * th)   # AdamW: decay applied to the weight
        else:
            th = th - lr * adaptive               # Adam+L2: decay already in the gradient
        coord1.append(round(th[1].item(), 4))
    return coord1, th

### Step 2 — Run both modes and compare the small-gradient weight

Now we run AdamW and Adam+L2 and read off the final value of the tiny-gradient weight. Under Adam+L2 that weight's decay gets divided by its small `sqrt(vhat)`... wait — its gradient is tiny, so `vhat` is tiny, and the coupled decay is effectively scaled in a way that barely shrinks it. AdamW applies a clean uniform shrink instead, regularizing the same weight far more aggressively by step 40.

In [ ]:
adamw, th_w = run(True)
adaml2, th_l = run(False)

print("AdamW   small-grad weight final:", th_w[1].item())   # ~ -1.3034
print("Adam+L2 small-grad weight final:", th_l[1].item())   # ~ -0.0983
print("AdamW   regularizes it ~13x more by step 40.")

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Do one AdamW step on a scalar weight $\theta=2.0$ with loss gradient $g=0.5$, defaults $\alpha=0.1$, $\beta_1=0.9$, $\beta_2=0.999$, $\epsilon\approx0$, $\lambda=0.1$, first step ($t=1$).

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- First moment: $m_1=(1-0.9)(0.5)=0.05$. — _Exponential average of the gradient, from $m_0=0$._
- Second moment: $v_1=(1-0.999)(0.25)=0.00025$. — _Exponential average of the squared gradient._
- Bias-correct: $\hat m_1=0.05/0.1=0.5$, $\hat v_1=0.00025/0.001=0.25$. — _Undo the cold-start shrink ($m,v$ began at 0)._
- Adaptive step: $\hat m_1/\sqrt{\hat v_1}=0.5/0.5=1.0$. — _On the first step the adaptive step is $\pm1$ in the gradient's direction._
- Decoupled decay: $\lambda\theta=0.1\times2.0=0.2$. — _Applied to the weight directly, not through $m,v$._
- Update: $\theta_{\text{new}}=2.0-0.1(1.0+0.2)=2.0-0.12=1.88$. — _Adaptive step plus uniform shrink, scaled by $\alpha$._

**Answer:** $\theta_{\text{new}}=1.88$. The CODE cell can confirm this against torch.optim.AdamW on the same one step.

</details>

**Problem 2.** Ablation: take the from-scratch AdamW in the CODE and fold the decay into the gradient instead (set $g\leftarrow g+\lambda\theta$, drop the separate $\lambda\theta$ term). What changes, and does the allclose still pass?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Move $\lambda\theta$ from the final update into the gradient before computing $m,v$. — _That converts AdamW back into Adam-with-L2._
- Re-run the allclose against torch.optim.AdamW. — _AdamW does NOT fold decay into the gradient, so the two now differ._
- Inspect a weight that saw large gradients vs one that saw tiny gradients. — _Under L2 the large-$\hat v$ weight is decayed less (Proposition 2)._

**Answer:** The allclose vs AdamW now fails &mdash; you have built Adam-with-L2, a different optimizer. The decay term is now divided by $\sqrt{\hat v_t}$, so weights with large gradient history get regularized less. (It would instead match torch.optim.Adam(weight_decay=...).) This is exactly the coupling the paper removes.

</details>

**Problem 3.** Why does the paper say decoupling makes the best $\lambda$ independent of the learning rate $\alpha$?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Under L2, the decay enters the loss gradient and is then scaled by both the adaptive term and $\alpha$, entangling $\lambda$ with $\alpha$ and with $\hat v_t$. — _Changing $\alpha$ changes the effective decay, so they must be tuned together._
- Under AdamW, the decay term is a clean $\eta_t\,\alpha\,\lambda\,\theta$ applied uniformly. — _The shape of the decay no longer depends on per-weight gradient history._

**Answer:** With L2 the effective regularization strength depends on $\alpha$ and on each weight's $\hat v_t$, so the optimal $\lambda$ shifts whenever you change the learning rate &mdash; forcing a 2-D hyperparameter search. Decoupling makes the decay a uniform, predictable shrink, so $\lambda$ and $\alpha$ can be tuned on separate axes (a practical headline of Section 4).

</details>